# Execution-time comparison: LIO-SAM vs FAST-LIO

This notebook automatically loads the logs for all four sensors and produces **seven figures**:

1. LIO-SAM: feature extraction, map optimization, total;
2. FAST-LIO: preprocess, mapping, total;
3. a total-time comparison of both algorithms, with one panel per sensor.

> **Definition of total:** the sum of the timings from both stages, paired in log order. This is not end-to-end latency: the CSV files contain neither queue wait times nor a unique scan identifier. If the stage sample counts differ, the notebook issues a warning and uses only the available pairs.

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")

WORKSPACE = Path("/home/neo/workspace")
LOG_DIR = WORKSPACE / "logs"

SENSORS = {
    "Livox MID-360": {
        "lio_sam": "lio_sam_livox_mid-360_ramp.csv",
        "fast_lio": "fast_lio_livox_mid-360_ramp.csv",
    },
    "Ouster OS1-64": {
        "lio_sam": "lio_sam_ouster_os1-64_ramp.csv",
        "fast_lio": "fast_lio_ouster_os1-64_ramp.csv",
    },
    "RoboSense RS-Helios-5515": {
        "lio_sam": "lio_sam_robosense_rs-helios-5515.csv",
        "fast_lio": "fast_lio_robosense_rs-helios-5515_ramp.csv",
    },
    "Velodyne VLP-16": {
        "lio_sam": "lio_sam_velodyne_vlp-16_ramp.csv",
        "fast_lio": "fast_lio_velodyne_vlp-16_ramp.csv",
    },
}

ALGORITHMS = {
    "lio_sam": {
        "label": "LIO-SAM",
        "stages": ("feature_extraction", "map_optimization"),
    },
    "fast_lio": {
        "label": "FAST-LIO",
        "stages": ("preprocess", "mapping"),
    },
}

SENSOR_COLORS = dict(zip(SENSORS, sns.color_palette("colorblind", len(SENSORS))))
ALGORITHM_COLORS = {
    "lio_sam": sns.color_palette("colorblind")[0],
    "fast_lio": sns.color_palette("colorblind")[1],
}

In [ ]:
REQUIRED_COLUMNS = {"timestamp", "thread", "process_time_ms"}
raw_data = {algorithm: {} for algorithm in ALGORITHMS}
total_data = {algorithm: {} for algorithm in ALGORITHMS}

def load_log(path, required_stages):
    if not path.is_file():
        raise FileNotFoundError(f"Log not found: {path}")

    frame = pd.read_csv(path)
    missing_columns = REQUIRED_COLUMNS.difference(frame.columns)
    if missing_columns:
        raise ValueError(f"{path.name}: missing columns {sorted(missing_columns)}")

    frame = frame.copy()
    frame["timestamp"] = pd.to_numeric(frame["timestamp"], errors="raise")
    frame["process_time_ms"] = pd.to_numeric(frame["process_time_ms"], errors="raise")
    frame["time_s"] = frame["timestamp"] - frame["timestamp"].min()

    missing_stages = set(required_stages).difference(frame["thread"].unique())
    if missing_stages:
        raise ValueError(f"{path.name}: missing stages {sorted(missing_stages)}")
    return frame

def build_total(frame, stages, source_name):
    stage_frames = []
    counts = {}
    for stage in stages:
        stage_frame = (
            frame.loc[frame["thread"] == stage, ["timestamp", "process_time_ms"]]
            .reset_index(drop=True)
            .rename(columns={
                "timestamp": f"timestamp_{stage}",
                "process_time_ms": stage,
            })
        )
        stage_frame.index.name = "sample"
        counts[stage] = len(stage_frame)
        stage_frames.append(stage_frame)

    if len(set(counts.values())) != 1:
        warnings.warn(
            f"{source_name}: stage sample counts differ {counts}; "
            "total uses only the available pairs in log order."
        )

    paired = pd.concat(stage_frames, axis=1, join="inner").dropna()
    paired["process_time_ms"] = paired[list(stages)].sum(axis=1)
    timestamp_columns = [f"timestamp_{stage}" for stage in stages]
    paired["timestamp"] = paired[timestamp_columns].max(axis=1)
    paired["time_s"] = paired["timestamp"] - frame["timestamp"].min()
    return paired.reset_index()

for sensor, files in SENSORS.items():
    for algorithm, settings in ALGORITHMS.items():
        path = LOG_DIR / files[algorithm]
        frame = load_log(path, settings["stages"])
        raw_data[algorithm][sensor] = frame
        total_data[algorithm][sensor] = build_total(frame, settings["stages"], path.name)

summary_rows = []
for algorithm, sensor_frames in raw_data.items():
    for sensor, frame in sensor_frames.items():
        for stage in ALGORITHMS[algorithm]["stages"]:
            values = frame.loc[frame["thread"] == stage, "process_time_ms"]
            summary_rows.append({
                "algorithm": ALGORITHMS[algorithm]["label"],
                "sensor": sensor,
                "stage": stage,
                "samples": len(values),
                "mean_ms": values.mean(),
                "median_ms": values.median(),
                "p95_ms": values.quantile(0.95),
            })
        values = total_data[algorithm][sensor]["process_time_ms"]
        summary_rows.append({
            "algorithm": ALGORITHMS[algorithm]["label"],
            "sensor": sensor,
            "stage": "total",
            "samples": len(values),
            "mean_ms": values.mean(),
            "median_ms": values.median(),
            "p95_ms": values.quantile(0.95),
        })

summary = pd.DataFrame(summary_rows)
summary.round(3)

In [ ]:
def plot_metric(algorithm, metric):
    algorithm_label = ALGORITHMS[algorithm]["label"]
    fig, ax = plt.subplots(figsize=(15, 6))

    for sensor in SENSORS:
        if metric == "total":
            frame = total_data[algorithm][sensor]
        else:
            frame = raw_data[algorithm][sensor]
            frame = frame.loc[frame["thread"] == metric]

        ax.plot(
            frame["time_s"],
            frame["process_time_ms"],
            label=sensor,
            color=SENSOR_COLORS[sensor],
            linewidth=1.1,
            alpha=0.85,
        )

    metric_label = metric.replace("_", " ").title()
    ax.set(
        title=f"{algorithm_label} - {metric_label}",
        xlabel="Elapsed dataset time [s]",
        ylabel="Execution time [ms]",
    )
    ax.legend(title="Sensor", ncols=2)
    ax.grid(True, alpha=0.25)
    fig.tight_layout()
    plt.show()

def plot_total_comparison():
    fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharex=False, sharey=True)

    for ax, sensor in zip(axes.flat, SENSORS):
        for algorithm, settings in ALGORITHMS.items():
            frame = total_data[algorithm][sensor]
            ax.plot(
                frame["time_s"],
                frame["process_time_ms"],
                label=settings["label"],
                color=ALGORITHM_COLORS[algorithm],
                linewidth=1.1,
                alpha=0.85,
            )
        ax.set_title(sensor)
        ax.set_xlabel("Elapsed time [s]")
        ax.set_ylabel("Total execution time [ms]")
        ax.grid(True, alpha=0.25)
        ax.legend()

    fig.suptitle("Total execution time: LIO-SAM vs FAST-LIO", fontsize=16)
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    plt.show()

## FAST-LIO — 3 plots

In [ ]:
for metric in ("preprocess", "mapping", "total"):
    plot_metric("fast_lio", metric)

## LIO-SAM — 3 plots

In [ ]:
for metric in ("feature_extraction", "map_optimization", "total"):
    plot_metric("lio_sam", metric)

## Total execution-time comparison — 1 plot

In [ ]:
plot_total_comparison()